# OTFLEX Gavin Workflow (Operation-Based Notebook)

This notebook executes the full experimental sequence directly in notebook cells, without graph traversal.

Execution model here:
- Each workflow action is a separate runnable code cell.
- Parameters are declared at the top of each action cell for easy edits.
- Cells are order-agnostic and can be rearranged.
- If something fails, jump to the teardown cell to leave devices in a safe state.

In [1]:
import asyncio
import json
import subprocess
import sys
import time
from pathlib import Path

# Find repo root so imports from src.* work no matter where notebook launches from.
repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent and not (repo_root / "src").exists():
    repo_root = repo_root.parent
if not (repo_root / "src").exists():
    raise RuntimeError("Could not find repository root containing src/")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.adapters.iot_mqtt import (
    PumpMQTT,
    UltraMQTT,
    HeatMQTT,
    ReactorMQTT,
    FurnaceMQTT,
    start_broker_if_needed,
    stop_broker,
    ControllerBeacon,
    _best_effort_all_off,
)
from src.adapters.otflex_adapter import OTFlex
from src.core.opentrons import opentronsClient

print(f"Repo root: {repo_root}")

# Device configs defined directly in this notebook
otflex_cfg = {
    "module": "otflex_runtime.py",
    "controller_ip": "192.168.10.161", # might be on 192.168.10.161, 169.254.179.32
    "deck": {
        "slots": {
            "A1": {"name": "single_electrode_module", "display": "3 electrodes", "labware": "sdl1_single_electrode_tiprack", "file": "data/labware_definitions/sdl1_single_electrode_tiprack.json", "slot_label": "A1"},
            "A2": {"name": "parallel_electrode_module", "display": "4 electrode tool", "labware": "sdl1_parallel_electrode_tiprack", "file": "data/labware_definitions/sdl1_parallel_electrode_tiprack.json", "slot_label": "A2"},
            "A3": {"name": "trash", "display": "trash", "labware": "opentrons_flex_trash", "slot_label": "A3"},
            "B1": {"name": "autreactor", "display": "autreactor", "labware": "actuated_reactor", "file": "data/labware_definitions/actuated_reactor.json", "slot_label": "B1"},
            "B2": {"name": "precursor_solutions", "display": "precursor solutions", "labware": "sdl1_11_vials_20mL", "file": "data/labware_definitions/sdl1_11_vials_20mL.json", "slot_label": "B2"},
            "B3": {"name": "sonicator_bath", "display": "sonicator bath", "labware": "nis_2_sonicator_bath", "file": "data/labware_definitions/nis_2_sonicator_bath.json", "slot_label": "B3"},
            "C3": {"name": "tiprack_1000ul", "display": "1000ul tips", "labware": "opentrons_flex_96_tiprack_1000ul", "slot_label": "C3"},
            "D1": {"name": "substrate_tower", "display": "tower of substrates", "labware": "zou_21_wellplate_4500ul", "file": "data/labware_definitions/zou_21_wellplate_4500ul.json", "slot_label": "D1"}
        },
        "pipettes": {"right": {"model": "p1000_single_flex"}}
    }
}

mqtt_cfg = {
    "broker": "localhost",
    "port": 1883,
    "username": "pyctl-controller",
    "password": "controller",
    "topics": {
        "pumps": "pumps/01",
        "reactor": "reactor/01",
        "furnace": "furnace/01",
        "heat": "heat/01",
        # Keep aligned with MQTT_Demo UltraMQTT base_topic.
        "ultra": "ultra/01",
    },
}

# Critical: flushWell uses OTFlex runtime MQTT helpers, so pass MQTT config into otflex runtime.
otflex_cfg["mqtt"] = mqtt_cfg

opentrons_cfg = {
    "ip": otflex_cfg["controller_ip"],
    "robot": "flex",
}

# Shared runtime objects.
otflex = OTFlex(otflex_cfg, root_dir=repo_root)

beacon = None
pumps = None
ultra = None
heat = None
reactor = None
furnace = None

print("Setup objects created. Run homing cell next, then connect devices.")

Repo root: C:\Users\Dell PC\Desktop\Projects\AC-OTFlex-monorepo
[OTFlex] Loaded module: C:\Users\Dell PC\Desktop\Projects\AC-OTFlex-monorepo\src\core\otflex_runtime.py
Setup objects created. Run homing cell next, then connect devices.


## Home Opentrons Robot First

Run this before any workflow action.

This cell inlines the same approach used by `scripts/home_opentrons.py` (directly using `opentronsClient`), without importing that script.

In [2]:
# Homing params (editable)
homing_params = {
    "ip": opentrons_cfg["ip"],
    "robot": opentrons_cfg["robot"],
}

client = opentronsClient(
    strRobotIP=homing_params["ip"],
    strRobot=homing_params["robot"],
)
client.homeRobot()
print(f"Homed {homing_params['robot']} at {homing_params['ip']}")

Homed flex at 192.168.10.161


## Capture Lab Setup Image (Before Connect)

Run this after homing and before connecting devices.

This uses the same SSH capture flow as the verbose Pi camera notebook:
- connect to Raspberry Pi over SSH
- capture an image with Picamera2
- download to `data/out/images`
- rotate the image 180 degrees
- display the image inline
- remove temporary remote image

In [ ]:
# Camera params (editable)
camera_params = {
    "host": "192.168.0.101",
    "username": "sdl1",
    "password": "1144",
    "ssh_port": 22,
    "connect_timeout_s": 8,
    "connect_retries": 3,
    "remote_capture_dir": "/tmp",
    "warmup_s": 2,
    "rotate_degrees": 180,
}

from datetime import datetime
import socket

try:
    import paramiko
except Exception as exc:
    raise ImportError(
        "paramiko is required for SSH camera capture. Install it in the notebook environment."
    ) from exc

try:
    from PIL import Image as PILImage
    from IPython.display import Image as IPyImage, display
except Exception as exc:
    raise ImportError(
        "Pillow and IPython display are required for rotate/display. Install pillow in the notebook environment."
    ) from exc

out_dir = repo_root / "data" / "out" / "images"
out_dir.mkdir(parents=True, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
filename = f"otflex-top_{timestamp}.jpg"
remote_path = f"{camera_params['remote_capture_dir']}/{filename}"
local_path = out_dir / filename

with socket.create_connection(
    (camera_params["host"], int(camera_params["ssh_port"])),
    timeout=3,
    ):
    pass
print(f"SSH port reachable at {camera_params['host']}:{camera_params['ssh_port']}")

ssh = None
try:
    ssh = paramiko.SSHClient()
    ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())

    for attempt in range(1, int(camera_params["connect_retries"]) + 1):
        try:
            print(f"SSH connect attempt {attempt}/{camera_params['connect_retries']}")
            ssh.connect(
                camera_params["host"],
                port=int(camera_params["ssh_port"]),
                username=camera_params["username"],
                password=camera_params["password"],
                timeout=float(camera_params["connect_timeout_s"]),
                banner_timeout=float(camera_params["connect_timeout_s"]),
                auth_timeout=float(camera_params["connect_timeout_s"]),
            )
            break
        except (socket.timeout, TimeoutError):
            if attempt == int(camera_params["connect_retries"]):
                raise
            print("Timeout; retrying...")

    capture_cmd = f"""python3 << 'EOF'
from picamera2 import Picamera2
import time

picam2 = Picamera2()
config = picam2.create_still_configuration(main={{\"size\": (2028, 1520)}})
picam2.configure(config)
picam2.start()
time.sleep({float(camera_params['warmup_s'])})
picam2.capture_file(\"{remote_path}\")
picam2.close()
print(\"OK\")
EOF
"""

    _, stdout, stderr = ssh.exec_command(capture_cmd, timeout=45)
    exit_code = stdout.channel.recv_exit_status()
    err_text = stderr.read().decode()
    if exit_code != 0:
        raise RuntimeError(f"Pi capture failed: {err_text}")

    sftp = ssh.open_sftp()
    sftp.get(remote_path, str(local_path))
    sftp.close()
    ssh.exec_command(f"rm {remote_path}")

    print(f"Lab setup image saved: {local_path}")

    # Rotate image to match top-down orientation used in verbose notebook.
    with PILImage.open(local_path) as img:
        rotated = img.rotate(int(camera_params["rotate_degrees"]), expand=True)
        rotated.save(local_path)

    print(f"Showing rotated image: {local_path.name}")
    display(IPyImage(filename=str(local_path)))
except Exception:
    raise
finally:
    if ssh is not None:
        ssh.close()

## Connect Devices

This cell mirrors MQTT_Demo MQTT client setup as closely as possible:
- connect OTFlex
- start controller beacon
- connect MQTT devices used by this workflow (pump/ultra/heat/reactor/furnace)
- use `pyctl-*` client IDs for consistent broker/client behavior

In [3]:
# Connection params (editable)
connect_params = {
    "settle_s": 0.8,
    "reset_existing_clients": True,
}

# Keep MQTT settings aligned with MQTT_Demo known-good values.
mqtt_cfg["broker"] = "192.168.0.100"
mqtt_cfg["port"] = 1883
mqtt_cfg["username"] = "pyctl-controller"
mqtt_cfg["password"] = "controller"
mqtt_cfg["topics"]["ultra"] = "ultra/01"

if connect_params["reset_existing_clients"]:
    # Re-running this cell with identical client IDs can cause broker-side disconnect thrash.
    for dev in (pumps, ultra, heat, reactor, furnace):
        if dev is not None:
            try:
                dev.disconnect()
            except Exception:
                pass
    if beacon is not None:
        try:
            beacon.stop()
        except Exception:
            pass

await otflex.connect()

beacon = ControllerBeacon(
    broker=mqtt_cfg["broker"],
    port=int(mqtt_cfg["port"]),
    username=mqtt_cfg["username"],
    password=mqtt_cfg["password"],
    client_id="pyctl-controller",
    status_topic="pyctl/status",
    heartbeat_topic="pyctl/heartbeat",
    heartbeat_interval=5.0,
    keepalive=30,
 )
beacon.start()

common = dict(
    broker=mqtt_cfg["broker"],
    port=int(mqtt_cfg["port"]),
    username=mqtt_cfg["username"],
    password=mqtt_cfg["password"],
)

pumps = PumpMQTT(**common, base_topic=mqtt_cfg["topics"]["pumps"], client_id="pyctl-pumps")
ultra = UltraMQTT(**common, base_topic=mqtt_cfg["topics"]["ultra"], client_id="pyctl-ultra")
heat = HeatMQTT(**common, base_topic=mqtt_cfg["topics"]["heat"], client_id="pyctl-heat")
reactor = ReactorMQTT(**common, base_topic=mqtt_cfg["topics"]["reactor"], client_id="pyctl-reactor")
furnace = FurnaceMQTT(**common, base_topic=mqtt_cfg["topics"]["furnace"], client_id="pyctl-furnace")

pumps.ensure_connected()
ultra.ensure_connected()
heat.ensure_connected()
reactor.ensure_connected()
furnace.ensure_connected()

# Ensure flushWell and other OTFlex MQTT helpers use known-good notebook clients.
rt = getattr(getattr(otflex, "mod", None), "_RT", None)
if rt is not None:
    rt.mqtt_pumps = pumps
    rt.mqtt_reactor = reactor
    rt.mqtt_furnace = furnace
    print("Bound OTFlex runtime MQTT handles to notebook MQTT clients.")

time.sleep(float(connect_params["settle_s"]))
print(f"MQTT connected: ultra client={ultra.client_id}, base={ultra.base}")
print("All configured devices connected.")

[OTFlex] Deck layout (normalized):
  slot A1 -> A1 : sdl1_single_electrode_tiprack (single_electrode_module)
  slot A2 -> A2 : sdl1_parallel_electrode_tiprack (parallel_electrode_module)
  slot A3 -> A3 : opentrons_flex_trash (trash)
  slot B1 -> B1 : actuated_reactor (autreactor)
  slot B2 -> B2 : sdl1_11_vials_20mL (precursor_solutions)
  slot B3 -> B3 : nis_2_sonicator_bath (sonicator_bath)
  slot C3 -> C3 : opentrons_flex_96_tiprack_1000ul (tiprack_1000ul)
  slot D1 -> D1 : zou_21_wellplate_4500ul (substrate_tower)
[otflex-pumps] Connecting to 192.168.0.100:1883 (attempt 1)...
[otflex-ultra] Connecting to 192.168.0.100:1883 (attempt 1)...
[otflex-pumps] Connected to 192.168.0.100:1883
[otflex-ultra] Connected to 192.168.0.100:1883
[otflex-heat] Connecting to 192.168.0.100:1883 (attempt 1)...
[otflex-heat] Connected to 192.168.0.100:1883
[otflex-reactor] Connecting to 192.168.0.100:1883 (attempt 1)...
[otflex-furnace] Connecting to 192.168.0.100:1883 (attempt 1)...
[otflex-reactor] 

## Raise Autoreactor (mqttReactor)

Publishes `FORWARD:5000` to the reactor topic and waits for firmware auto-stop timing.

In [ ]:
# Action params (editable)
params = {
    "direction": "forward",
    "duration_ms": 5000,
}

if params["direction"].lower() == "forward":
    await asyncio.to_thread(reactor.forward, params["duration_ms"])
else:
    await asyncio.to_thread(reactor.reverse, params["duration_ms"])
await asyncio.sleep(max(0.0, float(params["duration_ms"]) / 1000.0))
print("Reactor raise action complete.")

## Run xArm Plate-to-Reactor Transfer

This action replaces the fixed pause.

The notebook waits until `src/core/plate2reactor.py` completes, then the next action can lower the reactor.

In [ ]:
# Action params (editable)
params = {
    "script_relpath": "src/core/plate2reactor.py",
    "python_exe": sys.executable,
    "timeout_s": 0,  # 0 means no timeout
}

script_path = (repo_root / params["script_relpath"]).resolve()
if not script_path.exists():
    raise FileNotFoundError(f"xArm script not found: {script_path}")

cmd = [params["python_exe"], str(script_path)]
print("Running xArm transfer script:", " ".join(cmd))

run_kwargs = dict(
    cwd=str(repo_root),
    text=True,
    capture_output=True,
    check=False,
    timeout=None if int(params["timeout_s"]) <= 0 else float(params["timeout_s"]),
)

try:
    result = subprocess.run(cmd, **run_kwargs)
except subprocess.TimeoutExpired as exc:
    raise TimeoutError(
        f"xArm transfer timed out after {params['timeout_s']}s: {script_path}"
    ) from exc

if result.stdout:
    print(result.stdout)
if result.stderr:
    print(result.stderr)

if result.returncode != 0:
    raise RuntimeError(
        f"xArm transfer failed with exit code {result.returncode}: {script_path}"
    )

print("xArm transfer action complete.")

## Lower Autoreactor (mqttReactor)

Publishes `REVERSE:8000` and waits for duration alignment.

In [ ]:
# Action params (editable)
params = {
    "direction": "reverse",
    "duration_ms": 8000,
}

if params["direction"].lower() == "forward":
    await asyncio.to_thread(reactor.forward, params["duration_ms"])
else:
    await asyncio.to_thread(reactor.reverse, params["duration_ms"])
await asyncio.sleep(max(0.0, float(params["duration_ms"]) / 1000.0))
print("Reactor lower action complete.")

## Transfer Precursor A1 (otflexTransfer)

Transfers precursor from `precursor_solutions.A1` to reactor wells `A1` then `B1`.

In [ ]:
# Action params (editable)
params = {
    "from": {"labware": "precursor_solutions", "well": "A1", "offsetX": 0, "offsetY": 0, "offsetZ": 5},
    "to": {"labware": "autreactor", "well": ["A1", "B1"], "offsetX": 0, "offsetY": 0, "offsetZ": 20},
    "volume_uL": 500,
    "move_speed": 120,
    "pipette": "p1000_single_flex",
    "tiprack": "tiprack_1000ul",
    "autopick_tip": True,
}

await otflex.transfer(params)
print("Transfer action complete.")

## Transfer Precursor A2 (otflexTransfer)

Transfers precursor from `precursor_solutions.A2` to reactor wells `A1` then `B1`.

In [ ]:
# Action params (editable)
params = {
    "from": {"labware": "precursor_solutions", "well": "A2", "offsetX": 0, "offsetY": 0, "offsetZ": 5},
    "to": {"labware": "autreactor", "well": ["A1", "B1"], "offsetX": 0, "offsetY": 0, "offsetZ": 20},
    "volume_uL": 500,
    "move_speed": 120,
    "pipette": "p1000_single_flex",
    "tiprack": "tiprack_1000ul",
    "autopick_tip": True,
}

await otflex.transfer(params)
print("Transfer action complete.")

## ELECTRODEPOSITION - Operation Context

Initializes shared runtime handles and state used by the electrodeposition operations below.

Run this cell once before running any electrodeposition operation cells.

In [5]:
# Electrochem operation context (shared state + runtime handles)
rt = getattr(getattr(otflex, "mod", None), "_RT", None)
if rt is None or getattr(rt, "oc", None) is None:
    raise RuntimeError("OTFlex runtime controller is not available. Run setup/connect cells first.")

if "ec_state" not in globals():
    ec_state = {
        "tool_attached": False,
        "current_well": None,
        "at_ultrasonic": False,
    }

# Default locations used by operation cells; edit as needed.
ec_defaults = {
    "source_labware": "single_electrode_module",
    "source_well": "A1",
    "reactor_labware": "autreactor",
    "ultra_slot": "B3",
    "sonicator_labware": "sonicator_bath",
    "sonicator_well_a": "A1",
    "sonicator_well_b": "A2",
    "sonicator_immersion_mm": 35.0,
    "pipette": "p1000_single_flex",
}

print("Electrochem operation context ready.")
print(f"State: {ec_state}")

Electrochem operation context ready.
State: {'tool_attached': False, 'current_well': None, 'at_ultrasonic': False}


## ELECTRODEPOSITION - Pick Up Electrode Tool

Picks the electrode tool from its storage location and marks the system as tool-attached.

Use this before any well placement, deposition, or ultrasonic cleaning operations.

In [6]:
# Pick electrode tool from source rack
params = {
    "source_labware": ec_defaults["source_labware"],
    "source_well": ec_defaults["source_well"],
    "offsetX": 0.0,
    "offsetY": 0.0,
    "offsetZ": 0.0,
    "pipette": ec_defaults["pipette"],
}

rt = getattr(getattr(otflex, "mod", None), "_RT", None)
if rt is None or getattr(rt, "oc", None) is None:
    raise RuntimeError("OTFlex runtime controller is not available. Run setup/connect cells first.")

src_lw = rt.lw_ids.get(params["source_labware"], params["source_labware"])

await asyncio.to_thread(
    rt.oc.pickUpTip,
    strLabwareName=src_lw,
    strPipetteName=params["pipette"],
    strWellName=params["source_well"],
    fltOffsetX=float(params["offsetX"]),
    fltOffsetY=float(params["offsetY"]),
    fltOffsetZ=float(params["offsetZ"]),
)

ec_state["tool_attached"] = True
ec_state["current_well"] = None
ec_state["at_ultrasonic"] = False

print(f"Tool picked from {params['source_labware']}.{params['source_well']}")

Tool picked from single_electrode_module.A1


## ELECTRODEPOSITION - Move Tool To Reactor Well

Moves the attached electrode tool to the selected reactor well and positions it for in-well processing.

Use this whenever changing deposition target wells.

In [7]:
# Move attached electrode tool into a reactor well
params = {
    "reactor_labware": ec_defaults["reactor_labware"],
    "target_well": "C4",
    "offsetX": 0.0,
    "offsetY": 0.0,
    "descent_at_well_mm": 25.0,
    "move_speed": 80,
    "pipette": ec_defaults["pipette"],
}

if not ec_state.get("tool_attached", False):
    raise RuntimeError("No tool attached. Run the pick-tool cell first.")

rt = getattr(getattr(otflex, "mod", None), "_RT", None)
if rt is None or getattr(rt, "oc", None) is None:
    raise RuntimeError("OTFlex runtime controller is not available. Run setup/connect cells first.")

dst_lw = rt.lw_ids.get(params["reactor_labware"], params["reactor_labware"])
descent_mm = float(params["descent_at_well_mm"])
if descent_mm < 0:
    raise ValueError("descent_at_well_mm must be >= 0")

await asyncio.to_thread(
    rt.oc.moveToWell,
    strLabwareName=dst_lw,
    strWellName=params["target_well"],
    strPipetteName=params["pipette"],
    strOffsetStart="top",
    fltOffsetX=float(params["offsetX"]),
    fltOffsetY=float(params["offsetY"]),
    fltOffsetZ=0.0,
    intSpeed=int(params["move_speed"]),
)
await asyncio.to_thread(
    rt.oc.moveToWell,
    strLabwareName=dst_lw,
    strWellName=params["target_well"],
    strPipetteName=params["pipette"],
    strOffsetStart="top",
    fltOffsetX=float(params["offsetX"]),
    fltOffsetY=float(params["offsetY"]),
    fltOffsetZ=-descent_mm,
    intSpeed=int(params["move_speed"]),
)

ec_state["current_well"] = params["target_well"]
ec_state["at_ultrasonic"] = False
print(f"Tool positioned in {params['reactor_labware']}.{params['target_well']} at {descent_mm:.1f} mm depth")

Tool positioned in autreactor.C4 at 25.0 mm depth


## ELECTRODEPOSITION - Run In-Well Deposition

Runs the electrochemistry action while the electrode is already positioned in the reactor well.

This cell is the placeholder location for your final deposition logic.

In [48]:
import csv
import json
import os
from datetime import datetime

import paramiko

# Connection params to Biologic host (Windows host running Biologic stack)
biologic_params = {
    "host": "192.168.0.102",          # SSH host (Windows machine)
    "device_address": "192.168.0.102", # Potentiostat address for easy_biologic (set to instrument IP if different)
    "username": "sdl1_",
    "ssh_port": 22,
    "key_file": None,  # Set to path of private key, or None to auto-detect
    "channel": 4,  # Requested channel index from your workflow
}

# Experiment params - Chronoamperometry (CA) with capacity cutoff
experiment_params = {
    "potential_v": 1.5,
    "step_duration_s": 60,
    "cutoff_capacity_mah_cm2": 10.0,
    "electrode_area_cm2": 1.0,
}

# Create data output directory with timestamp
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
data_dir = repo_root / "data" / "out" / f"experiment_{timestamp}"
data_dir.mkdir(parents=True, exist_ok=True)

# Find SSH key file (look in common locations)
key_file = biologic_params.get("key_file")
if key_file is None:
    for potential_key in [
        os.path.expanduser("~/.ssh/id_rsa"),
        os.path.expanduser("~/.ssh/id_ed25519"),
        os.path.expanduser("~\\.ssh\\id_rsa"),
    ]:
        if os.path.exists(potential_key):
            key_file = potential_key
            break

if key_file is None:
    raise RuntimeError("SSH key not found. Set key_file in biologic_params or place key in ~/.ssh/")

print(f"Using SSH key: {key_file}")

# Establish SSH connection to Biologic host
print("\nConnecting to Biologic host via SSH...")
ssh = paramiko.SSHClient()
ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())

try:
    ssh.connect(
        biologic_params["host"],
        port=biologic_params["ssh_port"],
        username=biologic_params["username"],
        key_filename=key_file,
        timeout=10,
    )
    print(f"[OK] SSH connected to {biologic_params['username']}@{biologic_params['host']}")
except Exception as e:
    ssh = None
    raise RuntimeError(f"Failed to connect to Biologic host: {e}")

# Store SSH connection in global scope for use in execution cell
_biologic_ssh = ssh

# Remote script: tries `biologic` backend first, then `easy_biologic` fallback.
biologic_script = f"""
import json
import subprocess
import sys
import time
import traceback

HOST = {biologic_params['device_address']!r}
REQ_CHANNEL = int({biologic_params['channel']})
POTENTIAL_V = float({experiment_params['potential_v']})
STEP_DURATION_S = float({experiment_params['step_duration_s']})
CUTOFF_MAH_CM2 = float({experiment_params['cutoff_capacity_mah_cm2']})
ELECTRODE_AREA_CM2 = float({experiment_params['electrode_area_cm2']})


def emit_json(payload):
    print("===START_JSON_OUTPUT===")
    print(json.dumps(payload, indent=2))
    print("===END_JSON_OUTPUT===")


def err_out(msg):
    emit_json({{
        "data": [],
        "metadata": {{
            "status": "error",
            "error": msg,
        }},
    }})


def try_import_easy():
    import easy_biologic as ebl
    import easy_biologic.base_programs as blp
    return ebl, blp


backend = None
ebl = None
blp = None

# Backend 1: proprietary `biologic` package
try:
    from biologic import connect, I_RANGE
    from biologic.techniques.ca import CATechnique, CAParams, CAStep
    backend = "biologic"
except Exception:
    pass

# Backend 2: `easy_biologic` package (pre-installed or pip-installed)
if backend is None:
    try:
        ebl, blp = try_import_easy()
        backend = "easy_biologic"
    except Exception as pre_import_exc:
        print(f"[Biologic] easy_biologic import failed before install: {{pre_import_exc}}")
        traceback.print_exc()
        print("[Biologic] easy_biologic not found, attempting install...")
        try:
            # Pin setuptools to legacy-compatible version that provides pkg_resources.
            subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", "pip", "wheel", "setuptools<81", "-q"])
            try:
                import pkg_resources
                print("[Biologic] pkg_resources import OK after setuptools install")
            except Exception as pkg_exc:
                print(f"[Biologic] pkg_resources still missing after setuptools install: {{pkg_exc}}")
                traceback.print_exc()

            subprocess.check_call([sys.executable, "-m", "pip", "install", "easy-biologic", "-q"])
            ebl, blp = try_import_easy()
            backend = "easy_biologic"
            print("[Biologic] easy_biologic installed successfully")
        except Exception as install_exc:
            print(f"[Biologic] easy_biologic install failed: {{install_exc}}")
            traceback.print_exc()

if backend is None:
    err_out(
        "No supported Biologic backend found. Tried imports: biologic, easy_biologic (including pip install easy-biologic)"
    )
    raise SystemExit(1)

print(f"[Biologic] Selected backend: {{backend}}")
print(f"[Biologic] Target instrument address: {{HOST}}")

try:
    data = []
    max_capacity = 0.0

    if backend == "biologic":
        print("[Biologic] Connecting via biologic.connect()...")
        devices = connect()
        print(f"[Biologic] Found {{len(devices)}} device(s)")
        if len(devices) == 0:
            raise RuntimeError("No Biologic potentiostat devices found")

        device = devices[0]
        print(f"[Biologic] Using device: {{device}}")

        ca_step = CAStep(
            potential=POTENTIAL_V,
            duration=STEP_DURATION_S,
        )
        ca_params = CAParams(
            steps=[ca_step],
            current_range=I_RANGE.AUTO,
        )
        ca_technique = CATechnique(ca_params)

        device.load_technique(ca_technique)
        device.start()

        while device.is_running():
            data_obj = device.get_data()
            if data_obj and hasattr(data_obj, "time") and hasattr(data_obj, "current"):
                times = data_obj.time
                currents = data_obj.current
                potentials = data_obj.potential if hasattr(data_obj, "potential") else None

                while len(data) < len(times):
                    idx = len(data)
                    elapsed = float(times[idx])
                    current_a = float(currents[idx])
                    potential_v = float(potentials[idx]) if potentials is not None else POTENTIAL_V

                    current_density = current_a / ELECTRODE_AREA_CM2 if ELECTRODE_AREA_CM2 > 0 else 0.0
                    capacity = (current_density * elapsed * 1000.0) / 3600.0
                    max_capacity = max(max_capacity, capacity)

                    data.append({{
                        "Time_s": round(elapsed, 4),
                        "I_A": round(current_a, 8),
                        "I_density_A_cm2": round(current_density, 8),
                        "Capacity_mAh_cm2": round(capacity, 6),
                        "E_V": round(potential_v, 6),
                    }})

                    if capacity >= CUTOFF_MAH_CM2:
                        print(f"[Biologic] Capacity cutoff reached: {{capacity:.4f}} >= {{CUTOFF_MAH_CM2}}")
                        device.stop()
                        break

            time.sleep(0.1)

        try:
            device.stop()
        except Exception:
            pass

    elif backend == "easy_biologic":
        print("[Biologic] Connecting via easy_biologic...")
        bl = ebl.BiologicDevice(HOST)

        # easy_biologic typically uses 0-based channel indices.
        candidate_channels = [REQ_CHANNEL, max(0, REQ_CHANNEL - 1)]
        ch = candidate_channels[0]
        try:
            plugged = getattr(bl, "plugged", None)
            if plugged:
                for c in candidate_channels:
                    if c in plugged:
                        ch = c
                        break
        except Exception:
            pass

        params = {{
            "voltages": [POTENTIAL_V],
            "durations": [STEP_DURATION_S],
            "vs_initial": False,
            "time_interval": 0.1,
            "current_interval": 0.001,
        }}

        ca = blp.CA(bl, params, channels=[ch])
        ca.run("data")

        ch_data = []
        try:
            if isinstance(ca.data, dict):
                ch_data = ca.data.get(ch, [])
                if not ch_data and len(ca.data) > 0:
                    ch_data = next(iter(ca.data.values()))
            else:
                ch_data = ca.data or []
        except Exception:
            ch_data = []

        for idx, datum in enumerate(ch_data):
            def pick(obj, names, default=None):
                for n in names:
                    if hasattr(obj, n):
                        try:
                            return float(getattr(obj, n))
                        except Exception:
                            pass
                return default

            elapsed = pick(datum, ["time", "elapsed_time", "elapsed", "t", "ElapsedTime"], default=idx * 0.1)
            current_a = pick(datum, ["current", "I", "i", "Current"], default=0.0)
            potential_v = pick(datum, ["voltage", "Ewe", "potential", "E", "Voltage"], default=POTENTIAL_V)

            current_density = current_a / ELECTRODE_AREA_CM2 if ELECTRODE_AREA_CM2 > 0 else 0.0
            capacity = (current_density * elapsed * 1000.0) / 3600.0
            max_capacity = max(max_capacity, capacity)

            data.append({{
                "Time_s": round(elapsed, 4),
                "I_A": round(current_a, 8),
                "I_density_A_cm2": round(current_density, 8),
                "Capacity_mAh_cm2": round(capacity, 6),
                "E_V": round(potential_v, 6),
            }})

            if capacity >= CUTOFF_MAH_CM2:
                print(f"[Biologic] Capacity cutoff reached during post-processing: {{capacity:.4f}} >= {{CUTOFF_MAH_CM2}}")
                break

    emit_json({{
        "data": data,
        "metadata": {{
            "status": "success",
            "backend": backend,
            "target_address": HOST,
            "total_points": len(data),
            "max_capacity_mah_cm2": round(max_capacity, 6),
            "cutoff_capacity_mah_cm2": CUTOFF_MAH_CM2,
            "technique": "Chronoamperometry",
        }},
    }})

except Exception as e:
    traceback.print_exc()
    err_out(f"Experiment failed on backend={{backend}} target={{HOST}}: {{e}}")
"""

print("\n[OK] Experiment setup complete")
print(f"  SSH Host: {biologic_params['host']}")
print(f"  Instrument Address: {biologic_params['device_address']}")
print(f"  Channel: {biologic_params['channel']}")
print(f"  Potential: {experiment_params['potential_v']} V")
print(f"  Duration: {experiment_params['step_duration_s']} s")
print(f"  Cutoff Capacity: {experiment_params['cutoff_capacity_mah_cm2']} mAh/cm^2")
print(f"  Output directory: {data_dir}")

Using SSH key: C:\Users\Dell PC/.ssh/id_ed25519

Connecting to Biologic host via SSH...
[OK] SSH connected to sdl1_@192.168.0.102

[OK] Experiment setup complete
  SSH Host: 192.168.0.102
  Instrument Address: 192.168.0.102
  Channel: 4
  Potential: 1.5 V
  Duration: 60 s
  Cutoff Capacity: 10.0 mAh/cm^2
  Output directory: C:\Users\Dell PC\Desktop\Projects\AC-OTFlex-monorepo\data\out\experiment_20260424_130114


In [49]:
# Execute the Biologic script on the remote host
print("\n" + "=" * 70)
print("ELECTRODEPOSITION - CycoAmperometry Experiment")
print("=" * 70)

# Get SSH connection from setup cell
ssh = None
if "_biologic_ssh" not in globals() or _biologic_ssh is None:
    raise RuntimeError("SSH connection not established. Run the setup cell first.")
ssh = _biologic_ssh

try:
    # Upload script via SFTP to remote home directory
    print("\n[1/4] Uploading experiment script to Biologic host...")

    # Get remote home directory
    stdin, stdout, stderr = ssh.exec_command(
        "python3 -c \"import os; print(os.path.expanduser('~'))\"",
        timeout=5,
    )
    home_dir = stdout.read().decode().strip()
    remote_script_path = f"{home_dir}/biologic_cycloamperometry_temp.py"

    # Upload script
    sftp = ssh.open_sftp()
    with sftp.file(remote_script_path, "w") as f:
        f.write(biologic_script)
    sftp.close()
    print(f"[OK] Script uploaded to: {remote_script_path}")

    # Execute the script on remote host
    print("\n[2/4] Executing experiment (this may take several minutes)...")
    print("-" * 70)

    exec_cmd = f"python3 {remote_script_path}"
    stdin, stdout, stderr = ssh.exec_command(exec_cmd, timeout=600)

    # Capture all output in real-time
    output_lines = []
    error_lines = []

    for line in stdout:
        line = line.rstrip()
        print(line)
        output_lines.append(line)

    for line in stderr:
        line = line.rstrip()
        if line:
            print(f"[STDERR] {line}")
            error_lines.append(line)

    exit_code = stdout.channel.recv_exit_status()
    print("-" * 70)
    print(f"\n[OK] Experiment completed with exit code: {exit_code}")

    # Clean up remote script file
    try:
        ssh.exec_command(f"rm {remote_script_path}", timeout=5)
    except Exception as cleanup_err:
        print(f"[WARN] Could not clean up remote script: {cleanup_err}")

    # Parse JSON output from experiment
    print("\n[3/4] Processing results...")
    output_text = "\n".join(output_lines)

    try:
        start_idx = output_text.find("===START_JSON_OUTPUT===")
        end_idx = output_text.find("===END_JSON_OUTPUT===")

        if start_idx >= 0 and end_idx > start_idx:
            json_str = output_text[start_idx + len("===START_JSON_OUTPUT===") : end_idx].strip()
            json_data = json.loads(json_str)
            print("[OK] Experiment data parsed successfully")
            print(
                f"  Total data points: {json_data.get('metadata', {}).get('total_points', len(json_data.get('data', [])))}"
            )
            print(
                f"  Max capacity reached: {json_data.get('metadata', {}).get('max_capacity_mah_cm2', 'N/A')} mAh/cm^2"
            )
        else:
            print("[WARN] No JSON output found in experiment results")
            json_data = {"metadata": {"status": "no_output"}, "data": []}
    except json.JSONDecodeError as e:
        print(f"[WARN] Failed to parse JSON: {e}")
        json_data = {"metadata": {"status": "parse_error"}, "data": []}

    # Save raw output log
    print("\n[4/4] Saving data files...")
    log_file = data_dir / "experiment_log.txt"
    with open(log_file, "w") as f:
        f.write("=== CycoAmperometry Experiment Log ===\n")
        f.write(f"Timestamp: {timestamp}\n")
        f.write(f"Host: {biologic_params['host']}\n")
        f.write(f"Channel: {biologic_params['channel']}\n")
        f.write("\n=== STDOUT ===\n")
        f.write(output_text)
        if error_lines:
            f.write("\n=== STDERR ===\n")
            f.write("\n".join(error_lines))
    print(f"[OK] Log file: {log_file}")

    # Save experiment data to CSV
    if json_data.get("data") and len(json_data["data"]) > 0:
        csv_file = data_dir / "experiment_data.csv"
        first_point = json_data["data"][0]
        fieldnames = list(first_point.keys())

        with open(csv_file, "w", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            for row in json_data["data"]:
                writer.writerow(row)

        print(f"[OK] CSV file: {csv_file}")
        print(f"  Records: {len(json_data['data'])}")
    else:
        print("[WARN] No data points to save to CSV")
        csv_file = None

    # Save metadata and parameters to JSON
    metadata_file = data_dir / "metadata.json"
    metadata = {
        "experiment_type": "Chronoamperometry",
        "timestamp": timestamp,
        "well": ec_state.get("current_well", "unknown"),
        "biologic_host": biologic_params["host"],
        "biologic_channel": biologic_params["channel"],
        "experiment_params": experiment_params,
        "results": json_data.get("metadata", {}),
    }
    with open(metadata_file, "w") as f:
        json.dump(metadata, f, indent=2)
    print(f"[OK] Metadata file: {metadata_file}")

    print("\n" + "=" * 70)
    print("[OK] ELECTRODEPOSITION COMPLETE")
    print("=" * 70)
    print(f"\nData saved to: {data_dir}")
    print("\nFiles created:")
    print(f"  - {log_file.name}")
    if csv_file:
        print(f"  - {csv_file.name}")
    print(f"  - {metadata_file.name}")

except Exception as e:
    print(f"\n[ERROR] Experiment execution failed: {e}")
    import traceback

    traceback.print_exc()

finally:
    # Safely close SSH connection
    if ssh is not None:
        try:
            ssh.close()
            print("\n[OK] SSH connection closed")
        except Exception as close_err:
            print(f"[WARN] Error closing SSH connection: {close_err}")


ELECTRODEPOSITION - CycoAmperometry Experiment

[1/4] Uploading experiment script to Biologic host...
[OK] Script uploaded to: C:\Users\sdl1_/biologic_cycloamperometry_temp.py

[2/4] Executing experiment (this may take several minutes)...
----------------------------------------------------------------------
[Biologic] Selected backend: easy_biologic
[Biologic] Target instrument address: 192.168.0.102
[Biologic] Connecting via easy_biologic...
===START_JSON_OUTPUT===
{
  "data": [],
  "metadata": {
    "status": "error",
    "error": "Experiment failed on backend=easy_biologic target=192.168.0.102: ERR_COMM_CONNECTIONFAILED (-201): Could not establish communication with instrument."
  }
}
===END_JSON_OUTPUT===
[STDERR] C:\Users\sdl1_\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\easy_biologic\common.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resourc

## ELECTRODEPOSITION - Display Experiment Results

Visualizes the electrochemistry data collected during the CycoAmperometry experiment.

Generates plots of:
- Current vs Time
- Capacity vs Time
- Potential vs Time (if available)


In [16]:
# Display experiment results
import matplotlib.pyplot as plt
import pandas as pd
import json
from pathlib import Path
from datetime import datetime

# Find the most recent experiment data directory
out_base = repo_root / "data" / "out"
if not out_base.exists():
    print("No experiment data found.")
else:
    # Get list of timestamped directories, excluding 'images' if present
    exp_dirs = sorted([d for d in out_base.iterdir() if d.is_dir() and d.name != "images"], reverse=True)
    
    if not exp_dirs:
        print("No experiment data directories found.")
    else:
        # Use the most recent experiment
        exp_dir = exp_dirs[0]
        csv_file = exp_dir / "experiment_data.csv"
        metadata_file = exp_dir / "metadata.json"
        
        print(f"Loading experiment data from: {exp_dir.name}")
        
        if csv_file.exists() and metadata_file.exists():
            # Load data
            df = pd.read_csv(csv_file)
            with open(metadata_file) as f:
                metadata = json.load(f)
            
            print(f"Loaded {len(df)} data points")
            print(f"Columns: {', '.join(df.columns)}")
            
            # Create figure with subplots
            fig, axes = plt.subplots(2, 2, figsize=(14, 10))
            fig.suptitle(f"CycoAmperometry Experiment - {exp_dir.name}", fontsize=16, fontweight='bold')
            
            # Plot 1: Current vs Time
            if 'Time_s' in df.columns and 'I_A' in df.columns:
                axes[0, 0].plot(df['Time_s'], df['I_A'], 'b-', linewidth=1.5)
                axes[0, 0].set_xlabel('Time (s)', fontsize=11)
                axes[0, 0].set_ylabel('Current (A)', fontsize=11)
                axes[0, 0].set_title('Current vs Time', fontsize=12, fontweight='bold')
                axes[0, 0].grid(True, alpha=0.3)
            else:
                axes[0, 0].text(0.5, 0.5, 'Current data not available', ha='center', va='center')
                axes[0, 0].set_title('Current vs Time', fontsize=12, fontweight='bold')
            
            # Plot 2: Capacity vs Time
            if 'Time_s' in df.columns and 'Capacity_mAh_cm2' in df.columns:
                axes[0, 1].plot(df['Time_s'], df['Capacity_mAh_cm2'], 'g-', linewidth=1.5)
                axes[0, 1].set_xlabel('Time (s)', fontsize=11)
                axes[0, 1].set_ylabel('Capacity (mAh/cm²)', fontsize=11)
                axes[0, 1].set_title('Cumulative Capacity vs Time', fontsize=12, fontweight='bold')
                axes[0, 1].grid(True, alpha=0.3)
                
                # Add cutoff line if available
                if 'cutoff_capacity_mah_cm2' in metadata.get('experiment_params', {}):
                    cutoff = metadata['experiment_params']['cutoff_capacity_mah_cm2']
                    axes[0, 1].axhline(y=cutoff, color='r', linestyle='--', linewidth=2, label=f'Cutoff: {cutoff} mAh/cm²')
                    axes[0, 1].legend()
            else:
                axes[0, 1].text(0.5, 0.5, 'Capacity data not available', ha='center', va='center')
                axes[0, 1].set_title('Cumulative Capacity vs Time', fontsize=12, fontweight='bold')
            
            # Plot 3: Potential vs Time
            if 'Time_s' in df.columns and 'E_V' in df.columns:
                axes[1, 0].plot(df['Time_s'], df['E_V'], 'r-', linewidth=1.5)
                axes[1, 0].set_xlabel('Time (s)', fontsize=11)
                axes[1, 0].set_ylabel('Potential (V)', fontsize=11)
                axes[1, 0].set_title('Potential vs Time', fontsize=12, fontweight='bold')
                axes[1, 0].grid(True, alpha=0.3)
            else:
                axes[1, 0].text(0.5, 0.5, 'Potential data not available', ha='center', va='center')
                axes[1, 0].set_title('Potential vs Time', fontsize=12, fontweight='bold')
            
            # Plot 4: Metadata summary
            axes[1, 1].axis('off')
            summary_text = f"""
EXPERIMENT SUMMARY
{'─' * 40}
Type: {metadata.get('experiment_type', 'N/A')}
Timestamp: {metadata.get('timestamp', 'N/A')}
Well: {metadata.get('well', 'N/A')}
Data Points: {metadata.get('data_points', 'N/A')}

PARAMETERS
{'─' * 40}
Potential: {metadata.get('experiment_params', {}).get('potential_v', 'N/A')} V
Cycles: {metadata.get('experiment_params', {}).get('n_cycles', 'N/A')}
Step 1 Duration: {metadata.get('experiment_params', {}).get('step1_duration_s', 'N/A')} s
Step 2 Duration: {metadata.get('experiment_params', {}).get('step2_duration_s', 'N/A')} s
Cutoff Capacity: {metadata.get('experiment_params', {}).get('cutoff_capacity_mah_cm2', 'N/A')} mAh/cm²
Electrode Area: {metadata.get('experiment_params', {}).get('electrode_area_cm2', 'N/A')} cm²

DATA STATISTICS
{'─' * 40}
Total Points: {len(df)}
Time Range: {df['Time_s'].min():.2f} - {df['Time_s'].max():.2f} s
            """
            if 'I_A' in df.columns:
                summary_text += f"Current Range: {df['I_A'].min():.4f} - {df['I_A'].max():.4f} A\n"
            if 'Capacity_mAh_cm2' in df.columns:
                summary_text += f"Max Capacity: {df['Capacity_mAh_cm2'].max():.3f} mAh/cm²"
            
            axes[1, 1].text(0.05, 0.95, summary_text, transform=axes[1, 1].transAxes,
                           fontsize=9, verticalalignment='top', fontfamily='monospace',
                           bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.3))
            
            plt.tight_layout()
            plt.show()
            
            print(f"\n✓ Experiment data visualization complete")
        else:
            print(f"CSV file not found at: {csv_file}")


Loading experiment data from: 20260424_122702
CSV file not found at: C:\Users\Dell PC\Desktop\Projects\AC-OTFlex-monorepo\data\out\20260424_122702\experiment_data.csv


## ELECTRODEPOSITION - Move To Ultrasonic Station A

Relocates the attached electrode tool from reactor position to the first ultrasonic cleaning position.

Use this before triggering the first ultrasonic clean cycle.

In [ ]:
# Move tool to ultrasonic station A (well A1 in B3 sonicator bath)
params = {
    "ultra_slot": ec_defaults["ultra_slot"],
    "sonicator_labware": ec_defaults["sonicator_labware"],
    "target_well": ec_defaults["sonicator_well_a"],
    "offsetX": 0.0,
    "offsetY": 0.0,
    "immersion_depth_mm": ec_defaults["sonicator_immersion_mm"],
    "minimum_z_height": 80,
    "move_speed": 80,
    "pipette": ec_defaults["pipette"],
}

if not ec_state.get("tool_attached", False):
    raise RuntimeError("No tool attached. Run the pick-tool cell first.")

rt = getattr(getattr(otflex, "mod", None), "_RT", None)
if rt is None or getattr(rt, "oc", None) is None:
    raise RuntimeError("OTFlex runtime controller is not available. Run setup/connect cells first.")

# Safety first: always raise fully in Z before moving toward the sonicator bath.
import json
import requests

lift_command = {
    "data": {
        "commandType": "moveToAddressableArea",
        "params": {
            "minimumZHeight": int(params["minimum_z_height"]),
            "forceDirect": False,
            "speed": int(params["move_speed"]),
            "pipetteId": rt.oc.pipettes[params["pipette"]]["id"],
            "addressableAreaName": params["ultra_slot"],
            "stayAtHighestPossibleZ": True,
            "offset": {"x": 0.0, "y": 0.0, "z": 0.0},
        },
        "intent": "setup",
    }
}
lift_resp = requests.post(
    url=rt.oc.commandURL,
    headers=rt.oc.headers,
    params={"waitUntilComplete": True},
    data=json.dumps(lift_command),
)
if lift_resp.status_code != 201:
    raise RuntimeError(
        f"Failed to raise toolhead before sonicator move: {lift_resp.status_code} {lift_resp.text}"
    )

bath_lw = rt.lw_ids.get(params["sonicator_labware"], params["sonicator_labware"])
immersion_mm = float(params["immersion_depth_mm"])
if immersion_mm < 0:
    raise ValueError("immersion_depth_mm must be >= 0")

await asyncio.to_thread(
    rt.oc.moveToWell,
    strLabwareName=bath_lw,
    strWellName=params["target_well"],
    strPipetteName=params["pipette"],
    strOffsetStart="top",
    fltOffsetX=float(params["offsetX"]),
    fltOffsetY=float(params["offsetY"]),
    fltOffsetZ=0.0,
    intSpeed=int(params["move_speed"]),
)
await asyncio.to_thread(
    rt.oc.moveToWell,
    strLabwareName=bath_lw,
    strWellName=params["target_well"],
    strPipetteName=params["pipette"],
    strOffsetStart="top",
    fltOffsetX=float(params["offsetX"]),
    fltOffsetY=float(params["offsetY"]),
    fltOffsetZ=-immersion_mm,
    intSpeed=int(params["move_speed"]),
)

ec_state["at_ultrasonic"] = True
ec_state["current_well"] = None
print(f"Tool moved to {params['sonicator_labware']}.{params['target_well']} at {immersion_mm:.1f} mm immersion depth")

## ELECTRODEPOSITION - Ultrasonic Clean A

Triggers the first ultrasonic cleaning cycle while the electrode tool is held at the ultrasonic station.

Use this between deposition cycles to clean the electrode surface.

In [ ]:
# Run ultrasonic transducer (bath A)
params = {
    "channel": 1,
    "duration_s": 15.0,
    "use_auto_off_ms": True,
}

if not ec_state.get("at_ultrasonic", False):
    raise RuntimeError("Tool is not at ultrasonic position. Run a move-to-ultrasonic cell first.")

if ultra is None:
    raise RuntimeError("UltraMQTT is not connected. Run the Connect Devices cell first.")

duration_s = float(params["duration_s"])
if duration_s <= 0:
    raise ValueError("duration_s must be > 0")

channel = int(params["channel"])
duration_ms = max(1, int(round(duration_s * 1000)))

# Match MQTT_Demo behavior (ultra.on()/ultra.off()) and add defensive auto-off.
ultra.ensure_connected()
print(f"Ultrasonic MQTT client={ultra.client_id}, base={ultra.base}, channel={channel}")
if params["use_auto_off_ms"]:
    ultra.on(channel, duration_ms)
else:
    ultra.on(channel)
try:
    await asyncio.sleep(duration_s)
finally:
    # Keep explicit OFF for deterministic shutdown even if auto-off is enabled.
    ultra.off(channel)

print("Ultrasonic run complete.")

## ELECTRODEPOSITION - Move To Ultrasonic Station B

Relocates the attached electrode tool to the second ultrasonic cleaning position.

Use this when running multi-stage cleaning between deposition operations.

In [ ]:
# Move tool to ultrasonic station B (well A2 in B3 sonicator bath)
params = {
    "ultra_slot": ec_defaults["ultra_slot"],
    "sonicator_labware": ec_defaults["sonicator_labware"],
    "target_well": ec_defaults["sonicator_well_b"],
    "offsetX": 0.0,
    "offsetY": 0.0,
    "immersion_depth_mm": ec_defaults["sonicator_immersion_mm"],
    "minimum_z_height": 80,
    "move_speed": 80,
    "pipette": ec_defaults["pipette"],
}

if not ec_state.get("tool_attached", False):
    raise RuntimeError("No tool attached. Run the pick-tool cell first.")

rt = getattr(getattr(otflex, "mod", None), "_RT", None)
if rt is None or getattr(rt, "oc", None) is None:
    raise RuntimeError("OTFlex runtime controller is not available. Run setup/connect cells first.")

# Safety first: always raise fully in Z before moving toward the sonicator bath.
import json
import requests

lift_command = {
    "data": {
        "commandType": "moveToAddressableArea",
        "params": {
            "minimumZHeight": int(params["minimum_z_height"]),
            "forceDirect": False,
            "speed": int(params["move_speed"]),
            "pipetteId": rt.oc.pipettes[params["pipette"]]["id"],
            "addressableAreaName": params["ultra_slot"],
            "stayAtHighestPossibleZ": True,
            "offset": {"x": 0.0, "y": 0.0, "z": 0.0},
        },
        "intent": "setup",
    }
}
lift_resp = requests.post(
    url=rt.oc.commandURL,
    headers=rt.oc.headers,
    params={"waitUntilComplete": True},
    data=json.dumps(lift_command),
)
if lift_resp.status_code != 201:
    raise RuntimeError(
        f"Failed to raise toolhead before sonicator move: {lift_resp.status_code} {lift_resp.text}"
    )

bath_lw = rt.lw_ids.get(params["sonicator_labware"], params["sonicator_labware"])
immersion_mm = float(params["immersion_depth_mm"])
if immersion_mm < 0:
    raise ValueError("immersion_depth_mm must be >= 0")

await asyncio.to_thread(
    rt.oc.moveToWell,
    strLabwareName=bath_lw,
    strWellName=params["target_well"],
    strPipetteName=params["pipette"],
    strOffsetStart="top",
    fltOffsetX=float(params["offsetX"]),
    fltOffsetY=float(params["offsetY"]),
    fltOffsetZ=0.0,
    intSpeed=int(params["move_speed"]),
)
await asyncio.to_thread(
    rt.oc.moveToWell,
    strLabwareName=bath_lw,
    strWellName=params["target_well"],
    strPipetteName=params["pipette"],
    strOffsetStart="top",
    fltOffsetX=float(params["offsetX"]),
    fltOffsetY=float(params["offsetY"]),
    fltOffsetZ=-immersion_mm,
    intSpeed=int(params["move_speed"]),
)

ec_state["at_ultrasonic"] = True
ec_state["current_well"] = None
print(f"Tool moved to {params['sonicator_labware']}.{params['target_well']} at {immersion_mm:.1f} mm immersion depth")

## ELECTRODEPOSITION - Ultrasonic Clean B

Triggers the second ultrasonic cleaning cycle at the second cleaning position.

Use this for additional cleaning before returning to deposition or tool storage.

In [ ]:
# Run ultrasonic transducer (bath B)
params = {
    "channel": 1,
    "duration_s": 15.0,
    "use_auto_off_ms": True,
}

if not ec_state.get("at_ultrasonic", False):
    raise RuntimeError("Tool is not at ultrasonic position. Run a move-to-ultrasonic cell first.")

if ultra is None:
    raise RuntimeError("UltraMQTT is not connected. Run the Connect Devices cell first.")

duration_s = float(params["duration_s"])
if duration_s <= 0:
    raise ValueError("duration_s must be > 0")

channel = int(params["channel"])
duration_ms = max(1, int(round(duration_s * 1000)))

# Match MQTT_Demo behavior (ultra.on()/ultra.off()) and add defensive auto-off.
ultra.ensure_connected()
print(f"Ultrasonic MQTT client={ultra.client_id}, base={ultra.base}, channel={channel}")
if params["use_auto_off_ms"]:
    ultra.on(channel, duration_ms)
else:
    ultra.on(channel)
try:
    await asyncio.sleep(duration_s)
finally:
    # Keep explicit OFF for deterministic shutdown even if auto-off is enabled.
    ultra.off(channel)

print("Ultrasonic run complete.")

## ELECTRODEPOSITION - Return Tool

end electrodeposition

In [ ]:
# Move to another reactor well, or return tool to source
params = {
    "mode": "drop",  # "drop" or "next_well"
    "next_well": "B1",
    "reactor_labware": ec_defaults["reactor_labware"],
    "reactor_offsetX": 0.0,
    "reactor_offsetY": 0.0,
    "descent_at_well_mm": 25.0,
    "source_labware": ec_defaults["source_labware"],
    "source_well": ec_defaults["source_well"],
    "source_offsetX": 0.0,
    "source_offsetY": 0.0,
    "return_dZ": 12.0,
    "move_speed": 80,
    "pipette": ec_defaults["pipette"],
}

if not ec_state.get("tool_attached", False):
    raise RuntimeError("No tool attached. Run the pick-tool cell first.")

rt = getattr(getattr(otflex, "mod", None), "_RT", None)
if rt is None or getattr(rt, "oc", None) is None:
    raise RuntimeError("OTFlex runtime controller is not available. Run setup/connect cells first.")

if params["mode"] == "next_well":
    dst_lw = rt.lw_ids.get(params["reactor_labware"], params["reactor_labware"])
    descent_mm = float(params["descent_at_well_mm"])

    await asyncio.to_thread(
        rt.oc.moveToWell,
        strLabwareName=dst_lw,
        strWellName=params["next_well"],
        strPipetteName=params["pipette"],
        strOffsetStart="top",
        fltOffsetX=float(params["reactor_offsetX"]),
        fltOffsetY=float(params["reactor_offsetY"]),
        fltOffsetZ=0.0,
        intSpeed=int(params["move_speed"]),
    )
    await asyncio.to_thread(
        rt.oc.moveToWell,
        strLabwareName=dst_lw,
        strWellName=params["next_well"],
        strPipetteName=params["pipette"],
        strOffsetStart="top",
        fltOffsetX=float(params["reactor_offsetX"]),
        fltOffsetY=float(params["reactor_offsetY"]),
        fltOffsetZ=-descent_mm,
        intSpeed=int(params["move_speed"]),
    )
    ec_state["current_well"] = params["next_well"]
    ec_state["at_ultrasonic"] = False
    print(f"Tool moved to next well {params['reactor_labware']}.{params['next_well']}")
else:
    src_lw = rt.lw_ids.get(params["source_labware"], params["source_labware"])

    await asyncio.to_thread(
        rt.oc.moveToWell,
        strLabwareName=src_lw,
        strWellName=params["source_well"],
        strPipetteName=params["pipette"],
        strOffsetStart="top",
        fltOffsetX=float(params["source_offsetX"]),
        fltOffsetY=float(params["source_offsetY"]),
        fltOffsetZ=10,
        intSpeed=100,
    )
    await asyncio.to_thread(
        rt.oc.dropTip,
        strPipetteName=params["pipette"],
        boolDropInDisposal=False,
        strLabwareName=src_lw,
        strWellName=params["source_well"],
        strOffsetStart="bottom",
        fltOffsetX=float(params["source_offsetX"]),
        fltOffsetY=float(params["source_offsetY"]),
        fltOffsetZ=float(params["return_dZ"]),
    )

    ec_state["tool_attached"] = False
    ec_state["current_well"] = None
    ec_state["at_ultrasonic"] = False
    print(f"Tool dropped back at {params['source_labware']}.{params['source_well']}")

## Flush Tool Wells (otflexFlushWell)

Flushes wells `A1` and `B1` using pump IDs configured in params.
Each well flush now explicitly starts with OUT pump and ends with OUT pump.

In [ ]:
# Action params (editable)
params = {
    "from": {"labware": "single_electrode_module", "well": "C1", "offsetX": 0.0, "offsetY": 0.0, "offsetZ": 0.0},
    "to": {"labware": "autreactor", "well": ["A1", "B1"], "offsetX": 0, "offsetY": 0, "offsetZ": 5.0},
    "pipette": "p1000_single_flex",
    "in_pump_id": 2,
    "out_pump_id": 3,
    "in_time_ms": 2000,
    "out_time_ms": 5000,
    "repeats": 4,
    "purge_ms": 0,
    "return_dZ": 12.0,
}

# Preflight: flushWell relies on OTFlex runtime MQTT pump handle.
rt = getattr(getattr(otflex, "mod", None), "_RT", None)
if rt is None or rt.mqtt_pumps is None:
    raise RuntimeError(
        "OTFlex runtime MQTT pumps are not configured. Run the Connect Devices cell first."
    )

await otflex.flushWell(params)
print(
    f"Flush action complete (order per well: OUT -> (IN/OUT)x{params['repeats']} -> OUT; "
    f"in pump {params['in_pump_id']} for {params['in_time_ms']}ms, "
    f"out pump {params['out_pump_id']} for {params['out_time_ms']}ms)."
)

## Raise Autoreactor Final (mqttReactor)

Final reactor raise before furnace sequence.

In [ ]:
# Action params (editable)
params = {
    "direction": "forward",
    "duration_ms": 5000,
}

if params["direction"].lower() == "forward":
    await asyncio.to_thread(reactor.forward, params["duration_ms"])
else:
    await asyncio.to_thread(reactor.reverse, params["duration_ms"])
await asyncio.sleep(max(0.0, float(params["duration_ms"]) / 1000.0))
print("Reactor raise action complete.")

## Open Furnace Door (mqttFurnace)

Publishes `OPEN:6400` and waits to stay aligned with movement duration.

In [ ]:
# Action params (editable)
params = {
    "action": "open",
    "duration_ms": 6400,
}

if params["action"].lower() == "open":
    await asyncio.to_thread(furnace.open, params["duration_ms"])
else:
    await asyncio.to_thread(furnace.close, params["duration_ms"])
await asyncio.sleep(max(0.0, float(params["duration_ms"]) / 1000.0))
print("Furnace open action complete.")

## Run xArm Reactor-to-Furnace Transfer

This action replaces the fixed post-open pause.

The notebook waits until `src/core/reactor2furnace.py` completes, then the next action can close the furnace door.

In [ ]:
# Action params (editable)
params = {
    "script_relpath": "src/core/reactor2furnace.py",
    "python_exe": sys.executable,
    "timeout_s": 0,  # 0 means no timeout
}

script_path = (repo_root / params["script_relpath"]).resolve()
if not script_path.exists():
    raise FileNotFoundError(f"xArm script not found: {script_path}")

cmd = [params["python_exe"], str(script_path)]
print("Running xArm transfer script:", " ".join(cmd))

run_kwargs = dict(
    cwd=str(repo_root),
    text=True,
    capture_output=True,
    check=False,
    timeout=None if int(params["timeout_s"]) <= 0 else float(params["timeout_s"]),
)

try:
    result = subprocess.run(cmd, **run_kwargs)
except subprocess.TimeoutExpired as exc:
    raise TimeoutError(
        f"xArm transfer timed out after {params['timeout_s']}s: {script_path}"
    ) from exc

if result.stdout:
    print(result.stdout)
if result.stderr:
    print(result.stderr)

if result.returncode != 0:
    raise RuntimeError(
        f"xArm transfer failed with exit code {result.returncode}: {script_path}"
    )

print("xArm reactor-to-furnace action complete.")

## Close Furnace Door (mqttFurnace)

Publishes `CLOSE:10000` and waits to align with door motion.

In [ ]:
# Action params (editable)
params = {
    "action": "close",
    "duration_ms": 10000,
}

if params["action"].lower() == "open":
    await asyncio.to_thread(furnace.open, params["duration_ms"])
else:
    await asyncio.to_thread(furnace.close, params["duration_ms"])
await asyncio.sleep(max(0.0, float(params["duration_ms"]) / 1000.0))
print("Furnace close action complete.")

## End Marker

No hardware action. This marks the workflow end.

In [ ]:
print("Workflow sequence reached end marker.")

## Teardown - Safe Shutdown

Run this at the end, or immediately if any step errors.

This mirrors the runner's best-effort shutdown intent:
- turn off active MQTT outputs
- disconnect MQTT clients and beacon
- disconnect OTFlex

In [ ]:
try:
    _best_effort_all_off(pumps=pumps, ultra=ultra, heat=heat, reactor=reactor, furnace=furnace)
except Exception as exc:
    print(f"Best-effort OFF warning: {exc}")

for dev, name in ((pumps, "pumps"), (ultra, "ultra"), (heat, "heat"), (reactor, "reactor"), (furnace, "furnace")):
    if dev is not None:
        try:
            dev.disconnect()
        except Exception as exc:
            print(f"Disconnect warning ({name}): {exc}")

if beacon is not None:
    try:
        beacon.stop()
    except Exception as exc:
        print(f"Beacon stop warning: {exc}")

await otflex.disconnect()
print("Teardown complete.")